# Aggregate Figure 3

Generate the four real-dataset Figure 3 panels from source matrices and metadata.


## Setup

Load reusable figure assembly utilities and confirm required packages.


In [33]:
repo_root <- if (file.exists(file.path(getwd(), "evaluation_utils", "figure_aggregate_utils.R"))) {
  normalizePath(getwd(), mustWork = TRUE)
} else if (file.exists(file.path(getwd(), "..", "evaluation_utils", "figure_aggregate_utils.R"))) {
  normalizePath(file.path(getwd(), ".."), mustWork = TRUE)
} else {
  stop("Run this notebook from the repository root or from the evaluation/ directory.")
}

source(file.path(repo_root, "evaluation_utils", "figure_aggregate_utils.R"))
require_figure_packages()
source_plotting_utils(repo_root)


## Paths

Declare source data and output paths used by the aggregate figure.


In [34]:
figure3_output_format <- "png"
output_dir <- file.path(repo_root, "figures")
output_path <- file.path(output_dir, paste0("figure3.", figure3_output_format))

ecoli_data_path <- file.path(repo_root, "evaluation_data/ecoli")
ovarian_data_path <- file.path(repo_root, "evaluation_data/ovarian_cancer")
ccrcc_data_path <- file.path(repo_root, "evaluation_data/ccRCC_studies")
quartet_data_path <- file.path(repo_root, "evaluation_data/quartet_multiomics")

data.frame(
  role = c(
    "panel a source",
    "panel b source",
    "panel c source",
    "panel d source",
    "paper-ready figure output"
  ),
  path = c(
    ecoli_data_path,
    ovarian_data_path,
    ccrcc_data_path,
    quartet_data_path,
    output_path
  )
)


role,path
<chr>,<chr>
panel a source,/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/ecoli
panel b source,/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/ovarian_cancer
panel c source,/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/ccRCC_studies
panel d source,/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/quartet_multiomics
paper-ready figure output,/home/yuliya-cosybio/repos/cosybio/fedRBE/figures/figure3.png


## Labels And Style

Configure panel titles, palettes, model-term labels, and output size.


In [51]:

panel_titles <- c(
  ecoli = "a) E. coli (proteomics)",
  ovarian = "b) Ovarian cancer (microarrays)",
  ccrcc = "c) ccRCC (proteomics)",
  quartet = "d) Quartet (multi-omics)"
)

two_condition_palette <- c("#44abe7", "#c55702")
quartet_condition_palette <- c("#44abe7", "#c55702", "#009E73", "#F0E442")

common_term_labels <- c(
  condition = "Condition",
  lab = "Dataset",
  Status = "Status",
  Dataset = "Dataset"
)

figure_width_cm <- 18.5
figure_height_cm <- 18
cm_to_inches <- function(value_cm) value_cm / 2.54
figure_width <- cm_to_inches(figure_width_cm)
figure_height <- cm_to_inches(figure_height_cm)
figure_dpi <- 600
display_inline_preview <- FALSE


## Data Loaders

Read the before and after-BEC matrices used by the original real-dataset notebooks.


In [36]:

load_ecoli_data <- function(include_central = FALSE) {
  metadata <- utils::read.delim(
    file.path(ecoli_data_path, "before", "initial_data", "central_batch_info.tsv"),
    header = TRUE, row.names = 1, check.names = FALSE
  )
  metadata$file <- rownames(metadata)

  align_matrices_to_metadata(
    uncorrected = read_matrix_table(file.path(ecoli_data_path, "before", "central_intensities_log_UNION.tsv")),
    central_corrected = if (include_central) read_matrix_table(file.path(ecoli_data_path, "after", "intensities_log_Rcorrected_UNION.tsv")) else NULL,
    fed_corrected = read_matrix_table(file.path(ecoli_data_path, "after", "FedApp_corrected_data.tsv")),
    metadata = metadata,
    sample_col = "file"
  )
}

load_ovarian_data <- function(include_central = FALSE) {
  metadata <- utils::read.delim(
    file.path(ovarian_data_path, "before", "all_metadata.tsv"),
    header = TRUE, row.names = 1, check.names = FALSE
  )
  rownames(metadata) <- gsub("^X", "", rownames(metadata))
  metadata$file <- rownames(metadata)

  align_matrices_to_metadata(
    uncorrected = read_matrix_table(
      file.path(ovarian_data_path, "before", "all_expression_UNION.tsv"),
      zip_inner_file = "all_expression_UNION.tsv"
    ),
    central_corrected = if (include_central) read_matrix_table(file.path(ovarian_data_path, "after", "central_corrected_UNION.tsv")) else NULL,
    fed_corrected = read_matrix_table(file.path(ovarian_data_path, "after", "FedApp_corrected_data.tsv")),
    metadata = metadata,
    sample_col = "file"
  )
}

load_ccrcc_data <- function(include_central = FALSE) {
  metadata_raw <- utils::read.csv(
    file.path(ccrcc_data_path, "data", "ccRCC_metadata.csv"),
    header = TRUE, row.names = 1, check.names = FALSE
  )
  metadata <- data.frame(
    file = metadata_raw$Sample,
    condition = metadata_raw$Condition,
    lab = metadata_raw$Dataset,
    stringsAsFactors = FALSE
  )
  rownames(metadata) <- metadata$file

  align_matrices_to_metadata(
    uncorrected = read_matrix_table(file.path(ccrcc_data_path, "before", "central_intensities_log_UNION.tsv")),
    central_corrected = if (include_central) read_matrix_table(file.path(ccrcc_data_path, "after", "intensities_log_Rcorrected_UNION.tsv")) else NULL,
    fed_corrected = read_matrix_table(file.path(ccrcc_data_path, "after", "FedApp_corrected_data.tsv")),
    metadata = metadata,
    sample_col = "file"
  )
}

load_quartet_data <- function(include_central = FALSE) {
  donor_levels <- c("D5", "D6", "F7", "M8")
  metadata <- utils::read.delim(
    file.path(quartet_data_path, "all_modalities_metadata.tsv"),
    header = TRUE, check.names = FALSE
  )
  metadata$file <- metadata$pseudo_sample
  metadata$lab <- metadata$client
  metadata$condition <- factor(metadata$condition, levels = donor_levels)
  metadata$lab <- factor(metadata$lab)
  rownames(metadata) <- metadata$file

  fed_paths <- c(
    file.path(quartet_data_path, "after", "all_modalities_fedapp_kmeans_matrix.tsv"),
    file.path(quartet_data_path, "after", "all_modalities_fedsim_kmeans_matrix.tsv")
  )
  fed_path <- fed_paths[file.exists(fed_paths)][1]
  if (is.na(fed_path)) {
    stop("No Quartet federated all-modality matrix found.", call. = FALSE)
  }

  align_matrices_to_metadata(
    uncorrected = read_matrix_table(file.path(quartet_data_path, "before", "all_modalities_before_kmeans_matrix.tsv")),
    central_corrected = if (include_central) read_matrix_table(file.path(quartet_data_path, "after", "all_modalities_corrected_kmeans_matrix.tsv")) else NULL,
    fed_corrected = read_matrix_table(fed_path),
    metadata = metadata,
    sample_col = "file"
  )
}


## Load Data

Load and summarize each real-world dataset used in Figure 3.


In [37]:
figure3_data <- list(
  ecoli = prepare_plot_matrices(load_ecoli_data(include_central = FALSE)),
  ovarian = prepare_plot_matrices(load_ovarian_data(include_central = FALSE)),
  ccrcc = prepare_plot_matrices(load_ccrcc_data(include_central = FALSE)),
  quartet = prepare_plot_matrices(load_quartet_data(include_central = FALSE))
)

count_batches <- function(item) {
  if ("lab" %in% names(item$metadata)) {
    return(length(unique(item$metadata$lab)))
  }
  if ("Dataset" %in% names(item$metadata)) {
    return(length(unique(item$metadata$Dataset)))
  }
  NA_integer_
}

In [38]:
data.frame(
  dataset = names(figure3_data),
  features = vapply(figure3_data, function(item) nrow(item$uncorrected), integer(1)),
  plot_features = vapply(figure3_data, function(item) nrow(item$plot_matrices$uncorrected), integer(1)),
  samples = vapply(figure3_data, function(item) ncol(item$uncorrected), integer(1)),
  batches = vapply(figure3_data, count_batches, integer(1))
)

,dataset,features,plot_features,samples,batches
,<chr>,<int>,<int>,<int>,<int>
ecoli,ecoli,2702,2057,118,5
ovarian,ovarian,51276,21128,332,6
ccrcc,ccrcc,6282,2074,887,3
quartet,quartet,29517,29517,48,3


## Build Individual Panels

Define the shared plot style and dataset-specific plotting settings once, then build each dataset panel through the same local helper.


In [39]:
compact_figure_theme <- ggplot2::theme(
  plot.title = ggplot2::element_text(size = 7, margin = ggplot2::margin(b = 1.5)),
  axis.title = ggplot2::element_text(size = 6.8),
  axis.text = ggplot2::element_text(size = 6.2),
  legend.title = ggplot2::element_text(size = 6.5),
  legend.text = ggplot2::element_text(size = 5.9),
  legend.key.width = grid::unit(0.46, "cm"),
  legend.key.height = grid::unit(0.24, "cm"),
  legend.spacing.y = grid::unit(0.06, "cm"),
  legend.margin = ggplot2::margin(0, 0, 0, 0),
  plot.margin = ggplot2::margin(2, 2, 2, 5)
)

compact_figure_guides <- ggplot2::guides(
  color = ggplot2::guide_legend(override.aes = list(size = 1.45), title.position = "top"),
  shape = ggplot2::guide_legend(override.aes = list(size = 1.45), title.position = "top"),
  fill = ggplot2::guide_legend(
    title.position = "top",
    keywidth = grid::unit(0.46, "cm"),
    keyheight = grid::unit(0.24, "cm")
  )
)

variance_median_label_size <- 2.15

dataset_panel_order <- c("ecoli", "ovarian", "ccrcc", "quartet")

dataset_plot_specs <- list(
  ecoli = list(
    color_col = "condition",
    shape_col = "lab",
    palette = two_condition_palette,
    color_title = "Condition",
    shape_title = "Dataset",
    form = ~ condition + lab
  ),
  ovarian = list(
    color_col = "Status",
    shape_col = "Dataset",
    palette = two_condition_palette,
    color_title = "Status",
    shape_title = "Dataset",
    form = ~ Status + Dataset
  ),
  ccrcc = list(
    color_col = "condition",
    shape_col = "lab",
    palette = two_condition_palette,
    color_title = "Condition",
    shape_title = "Dataset",
    form = ~ condition + lab
  ),
  quartet = list(
    color_col = "condition",
    shape_col = "lab",
    palette = quartet_condition_palette,
    color_title = "Condition",
    shape_title = "Dataset",
    form = ~ condition + lab
  )
)

style_figure3_plot <- function(plot) {
  plot + compact_figure_theme + compact_figure_guides
}

build_dataset_panel <- function(dataset_key) {
  spec <- dataset_plot_specs[[dataset_key]]
  if (is.null(spec)) {
    stop("No Figure 3 plot specification for dataset: ", dataset_key, call. = FALSE)
  }

  dataset_data <- figure3_data[[dataset_key]]
  if (is.null(dataset_data)) {
    stop("No Figure 3 data loaded for dataset: ", dataset_key, call. = FALSE)
  }

  plot_matrices <- dataset_data$plot_matrices
  metadata <- dataset_data$metadata

  pca_before <- pca_plot(
    df = plot_matrices$uncorrected,
    batch_info = metadata,
    title = "Before correction",
    quantitative_col_name = "file",
    col_col = spec$color_col,
    shape_col = spec$shape_col,
    show_legend = FALSE,
    cbPalette = spec$palette,
    color_title = spec$color_title,
    shape_title = spec$shape_title,
    point_size = 0.8,
    use_gram = TRUE
  )

  pca_after <- pca_plot(
    df = plot_matrices$fed_corrected,
    batch_info = metadata,
    title = "After fedRBE",
    quantitative_col_name = "file",
    col_col = spec$color_col,
    shape_col = spec$shape_col,
    show_legend = TRUE,
    cbPalette = spec$palette,
    color_title = spec$color_title,
    shape_title = spec$shape_title,
    point_size = 0.8,
    use_gram = TRUE
  )

  variance_before <- lmpv_plot(
    data = plot_matrices$uncorrected,
    metadata = metadata,
    title = "Before correction",
    form = spec$form,
    show_legend = FALSE,
    term_labels = common_term_labels,
    fill_title = NULL,
    median_label_size = variance_median_label_size
  )

  variance_after <- lmpv_plot(
    data = plot_matrices$fed_corrected,
    metadata = metadata,
    title = "After fedRBE",
    form = spec$form,
    show_legend = TRUE,
    term_labels = common_term_labels,
    fill_title = NULL,
    median_label_size = variance_median_label_size
  )

  make_dataset_diagnostic_panel(
    title = panel_titles[[dataset_key]],
    pca_before = style_figure3_plot(pca_before),
    pca_after = style_figure3_plot(pca_after),
    variance_before = style_figure3_plot(variance_before),
    variance_after = style_figure3_plot(variance_after)
  )
}


## Assemble Panels

Build all four dataset panels before arranging them into the final two-column figure.


In [40]:
figure3_dataset_panels <- lapply(dataset_panel_order, build_dataset_panel)
names(figure3_dataset_panels) <- dataset_panel_order



## Aggregate Layout

Arrange the already-built dataset panels into the final two-column layout. Re-run this cell and the save cell to adjust layout or dimensions without recomputing PCA and variance plots.


In [47]:
figure3_layout <- arrange_dataset_panels(
  panels = figure3_dataset_panels,
  ncol = 2,
  title_fontsize = 8.8,
  subplot_row_gap = grid::unit(0.34, "cm"),
  subplot_column_gap = grid::unit(0.14, "cm"),
  legend_gap = grid::unit(0.18, "cm"),
  dataset_row_gap = grid::unit(0.34, "cm"),
  dataset_column_gap = grid::unit(0.18, "cm")
)

## Preview

Render the aggregate layout inline when running in an interactive notebook kernel.


In [48]:
if (figure_should_display_inline() && isTRUE(display_inline_preview)) {
  draw_dataset_panel_layout(figure3_layout)
} else {
  message("Inline display skipped. Set display_inline_preview <- TRUE to render before saving.")
}


Inline display skipped. Set display_inline_preview <- TRUE to render before saving.



## Save

Write the paper-ready aggregate Figure 3 image.


In [52]:
saved_path <- save_dataset_panel_layout(
  layout = figure3_layout,
  output_path = output_path,
  width = figure_width,
  height = figure_height * figure3_layout$nrow / 2,
  dpi = figure_dpi
)

cat("Saved Figure 3 to", saved_path, "\n")


Saved Figure 3 to /home/yuliya-cosybio/repos/cosybio/fedRBE/figures/figure3.png 
